# next-track-rec 02: SASRec retrieval

Runs code from github.com/N-Mehdi/next-track-rec on the private dataset yambda-50m-prepared.
Two configurations: A (embedding dim 128) and B (embedding dim 256). Outputs go to /kaggle/working/outputs.

In [ ]:
import glob
import os
import subprocess

REPO = "https://github.com/N-Mehdi/next-track-rec.git"
CODE = "/tmp/next-track-rec"
subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)

PREPARED = os.path.dirname(glob.glob("/kaggle/input/**/item_map", recursive=True)[0])
OUTPUTS = "/kaggle/working/outputs"
print(PREPARED)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
!python -m nexttrack.train.train_sasrec --set paths.prepared={PREPARED} paths.outputs={OUTPUTS} sasrec.run_name=a_d128 sasrec.embedding_dim=128

In [ ]:
!python -m nexttrack.train.train_sasrec --set paths.prepared={PREPARED} paths.outputs={OUTPUTS} sasrec.run_name=b_d256 sasrec.embedding_dim=256

In [ ]:
import json

for run in ["a_d128", "b_d256"]:
    with open(os.path.join(OUTPUTS, run, "report.json"), encoding="utf-8") as handle:
        report = json.load(handle)
    print(run, "best epoch", report["best_epoch"], "train seconds", round(report["train_seconds"]))
    print("  val ", report["validation"]["all"]["ndcg@10"], report["validation"]["all"]["recall@10"])
    print("  test", report["test"]["all"]["ndcg@10"], report["test"]["all"]["recall@10"])